# Descargar Video del Partido

Baja el video de un partido desde el link de la página del player y lo guarda
en la carpeta del partido (`data/temporadas/{año}/partidos/{match_key}/video/`),
listo para el pipeline de tracking (`analisis_video.ipynb` / `procesar_video_colab.ipynb`).

**Probado con [lpfplay.com](https://www.lpfplay.com)** (Liga Profesional / AFA), que
usa immergo.tv con **HLS sin DRM**. La URL del stream no está en el HTML — la pide
el player por JavaScript — así que se captura con un navegador headless leyendo el
tráfico de red, y se descarga con yt-dlp.

> Solo descargar contenido para el que tengas derecho de uso. Esto es para análisis
> propio del equipo.

In [ ]:
import os, sys

%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config
from src.video.downloader import capture_stream_url, download_match_video

# ============================================================
# CONFIGURAR ACÁ
PAGE_URL  = 'https://www.lpfplay.com/player/episode/REEMPLAZAR'
SEASON    = '2026'
MATCH_KEY = 'YYYY-MM-DD_vs_<rival>_<h|a>'
QUALITY   = '720p'    # '360p' | '720p' (recomendado CV) | '1080p'
HEADLESS  = True      # False si la captura falla y querés ver el navegador
# ============================================================

VIDEO_DIR = config.match_video_dir(MATCH_KEY, SEASON)
print(f'Partido : {MATCH_KEY}')
print(f'Calidad : {QUALITY}')
print(f'Destino : {VIDEO_DIR}')

## 1. Capturar la URL del stream

Abre la página en un navegador headless y lee el `master.m3u8` del tráfico de red.
Esto NO descarga nada todavía — solo confirma que el partido es accesible.

In [ ]:
manifest = capture_stream_url(PAGE_URL, headless=HEADLESS, verbose=True)
if manifest:
    print(f'\n🟢 Stream encontrado:\n{manifest}')
else:
    print('\n🔴 No se capturó el manifest. Probá HEADLESS=False o revisá el link.')

## 2. (Opcional) Prueba rápida — 1 fragmento

Antes de bajar el partido completo (varios GB en 720p), validá la cadena bajando
un solo fragmento. Si esto crea un `.mp4` chiquito, todo funciona.

In [ ]:
# Baja solo 1 fragmento usando el manifest ya capturado (test=True)
prueba = download_match_video(manifest, MATCH_KEY + '_PRUEBA', season=SEASON,
                             quality=QUALITY, test=True, verbose=True)
import os
print('OK' if os.path.exists(prueba) else 'FALLO')
# borrar el archivo de prueba
if os.path.exists(prueba):
    os.remove(prueba)
    print('(archivo de prueba eliminado)')

## 3. Descargar el partido completo

Baja el video entero a la carpeta del partido. **Puede tardar** (en 720p son
varios GB). Al terminar, el partido queda listo para `analisis_video.ipynb` /
`procesar_video_colab.ipynb`.

In [ ]:
# Usa el manifest ya capturado (más rápido que reabrir el navegador)
video_path = download_match_video(manifest, MATCH_KEY, season=SEASON,
                                  quality=QUALITY, verbose=True)
print(f'\n✅ Listo. Ahora podés correr analisis_video.ipynb / procesar_video_colab.ipynb con:')
print(f'   SEASON    = "{SEASON}"')
print(f'   MATCH_KEY = "{MATCH_KEY}"')